In [1]:
library(Seurat)

Warning message:
“package ‘Seurat’ was built under R version 4.2.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.2.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.2.3”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




In [2]:
input_files <- c('/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_Ire_cetuxi/CRC0322_NT_1_3000_dir/filtered_annotated_CRC0322_NT_1_3000.tsv',
                 '/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_Ire_cetuxi/CRC0327_NT_2_dir/filtered_annotated_CRC0327_NT_2.tsv', 
                 '/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_longer/CRC0542_NT72h_1_dir/filtered_annotated_CRC0542_NT72h_1.tsv')
                 
samples <- c('CRC0322', 'CRC0327', 'CRC0542')

#filtergenes <- ''
#or 
filtergenes <- 'Graphs/grafo_filtrato_undirect_small.csv'

if (filtergenes != '') {
    nfeatures <- 949
    npc <- 300
    resolution <- 0.3 # even with 0.8 we do not get a better cluster 4
} else {
    nfeatures <- 1000
    npc <- 20
    resolution <- 0.1
}

In [3]:
genes <- read.csv(filtergenes)
keepg <- unique(c(genes$source))
length(keepg)

[1] 949

In [4]:
# create seurat objects and integrates
input_files_l <- unlist(strsplit(input_files, ','))

createSeurat <- function(file, genes) {
  df <- read.table(gzfile(file), sep=',', header=TRUE, row.names=1)
  keep <- sapply(rownames(df), function(x) {g <- strsplit(x, ':')[1][[1]]; g[2] %in% genes})
  df <- df[keep,]
  res <- CreateSeuratObject(df)
  return(res)
}

data_list <- lapply(input_files_l, createSeurat, keepg)

data_list <- lapply(X = data_list, FUN = function(x) {
    x <- NormalizeData(x)
    x <- ScaleData(x)
    x <- FindVariableFeatures(x, selection.method = "vst", nfeatures = nfeatures)
    
})

#features <- SelectIntegrationFeatures(object.list = data_list, nfeatures= nfeatures)
#anchors <- FindIntegrationAnchors(object.list = data_list, anchor.features = features)#, dims=1:npc) paneth split in two clusters in this way
#sdata <- IntegrateData(anchorset = anchors)#, dims=1:npc) 
#DefaultAssay(sdata) <- "integrated"



Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”
Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”
Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”
Normalizing layer: counts

Finding variable features for layer counts

Normalizing layer: counts

Finding variable features for layer counts

Normalizing layer: counts

Finding variable features for layer counts

Warning message in CheckDuplicateCellNames(object.list = object.list):
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”
Scaling features for provided objects

Finding all pairwise anchors

Running CCA

Merging objects

Finding neighborhoods

Finding anchors

	Found 10925 anchors

Filtering anchors

	Retained 3463 anchors

Running CCA

Merging objects

Finding neighborhoods

Finding anchors

	Found 9951 anchors

Filtering anchors

	Retained 3324 anchors

Running CCA

Merging objects

Finding neighborhoods

Finding anchors

In [22]:
data_list <- lapply(X = data_list, FUN = function(x) {
    x <- NormalizeData(x)
    x <- ScaleData(x)
    #x <- RunPCA(x, npcs= npc, verbose=FALSE)
    x <- FindVariableFeatures(x, selection.method = "vst", nfeatures = nfeatures)
    
})

Normalizing layer: counts

Centering and scaling data matrix

Warning message:
“The following 10 features requested have zero variance; running reduction without them: ENSG00000140365:COMMD4, ENSG00000105738:SIPA1L3, ENSG00000168143:FAM83B, ENSG00000125733:TRIP10, ENSG00000131263:RLIM, ENSG00000180776:ZDHHC20, ENSG00000197217:ENTPD4, ENSG00000147459:DOCK5, ENSG00000110711:AIP, ENSG00000126267:COX6B1”
PC_ 1 
Positive:  ENSG00000089356:FXYD3, ENSG00000166165:CKB, ENSG00000133112:TPT1, ENSG00000172238:ATOH1, ENSG00000107954:NEURL1, ENSG00000275395:FCGBP, ENSG00000016490:CLCA1, ENSG00000188175:HEPACAM2, ENSG00000059728:MXD1, ENSG00000122711:SPINK4 
	   ENSG00000163586:FABP1, ENSG00000160593:JAML, ENSG00000151135:TMEM263, ENSG00000075426:FOSL2, ENSG00000165949:IFI27, ENSG00000105856:HBP1, ENSG00000172183:ISG20, ENSG00000126709:IFI6, ENSG00000169583:CLIC3, ENSG00000006459:KDM7A 
	   ENSG00000118507:AKAP7, ENSG00000101236:RNF24, ENSG00000177606:JUN, ENSG00000132549:VPS13B, ENSG00000205542:TMS

In [23]:
ser_merged <- merge(x=data_list[[1]], y=data_list[2:length(data_list)])

Warning message:
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”


In [ ]:
ser_merged <- RunPCA(ser_merged, npcs= npc, verbose=FALSE)

Warning message in LayerData.Assay5(object = object, layer = layer):
“multiple layers are identified by scale.data.1 scale.data.2 scale.data.3
 only the first layer is used”


In [26]:
sdata <- IntegrateLayers(object = ser_merged, method = CCAIntegration, orig.reduction = "pca", new.reduction="integrated.cca", verbose = FALSE)

ERROR: [1m[33mError[39m in `IntegrateLayers()`:[22m
[33m![39m ‘pca’ is not a dimensional reduction


In [24]:
data_list
ser_merged
sdata

[[1]]
An object of class Seurat 
949 features across 2906 samples within 1 assay 
Active assay: RNA (949 features, 949 variable features)
 3 layers present: counts, data, scale.data
 1 dimensional reduction calculated: pca

[[2]]
An object of class Seurat 
949 features across 4006 samples within 1 assay 
Active assay: RNA (949 features, 949 variable features)
 3 layers present: counts, data, scale.data
 1 dimensional reduction calculated: pca

[[3]]
An object of class Seurat 
949 features across 3138 samples within 1 assay 
Active assay: RNA (949 features, 949 variable features)
 3 layers present: counts, data, scale.data
 1 dimensional reduction calculated: pca


An object of class Seurat 
949 features across 10050 samples within 1 assay 
Active assay: RNA (949 features, 949 variable features)
 9 layers present: counts.1, counts.2, counts.3, data.1, scale.data.1, data.2, scale.data.2, data.3, scale.data.3

An object of class Seurat 
1898 features across 10050 samples within 2 assays 
Active assay: integrated (949 features, 949 variable features)
 2 layers present: data, scale.data
 1 other assay present: RNA

In [11]:
# Scale PCA UMAP on integrated
sdata <- ScaleData(sdata, verbose = FALSE)
sdata <- RunPCA(sdata, verbose = FALSE, npcs = npc)

ElbowPlot(sdata)

sdata <- RunUMAP(sdata, reduction = "pca", dims = 1:npc, verbose=F)
sdata <- FindNeighbors(sdata, reduction = "pca", dims = 1:npc, verbose=F)

In [ ]:
#UMAP with samples
cells <- colnames(sdata)

samples_n <- sapply(strsplit(cells, "_"), function(x){x[[2]]})
sdata$sample <- samples[as.numeric(samples_n)]
DimPlot(sdata, reduction = "umap", group.by='sample')


In [ ]:
# UMAP with clusters
DefaultAssay(sdata) <- "integrated"

sdata <- FindClusters(sdata, resolution = resolution)
DimPlot(sdata, reduction = "umap", label = TRUE, repel = TRUE)

In [ ]:
DefaultAssay(sdata) <- "RNA"
sdata <- JoinLayers(sdata)

df <- data.frame(row.names=names(sdata$orig.ident), sample=sdata$sample, cluster=sdata$seurat_clusters)

n_cl <- length(unique(df$cluster))
deg <- list()
for (i in seq(0,(n_cl-1))) {
	cl <- as.data.frame(FindConservedMarkers(sdata, ident.1 = i, verbose = FALSE, grouping.var="sample"))
    deg[[i+1]] <- cl[cl$max_pval < 0.05,]
}


In [ ]:
for (i in seq(0,(n_cl-1))) {
    print(paste0('cluster ', i))
    d <- deg[[i+1]]
    d <- d[order(-d$CRC0322_avg_log2FC),]
    print(head(d[, grepl('log2FC', colnames(d))], n=10))
    print(head(d[, grepl('adj', colnames(d))], n=10))
}

In [ ]:
filesp <- c('/mnt/cold1/snaketree/prj/scRNA/dataset/KMeans_2/kmeans/CRC0322_NT_1_3000/CRC0322_NT_1_3000_kmeans_2comp.csv',
            '/mnt/cold1/snaketree/prj/scRNA/dataset/KMeans_2/kmeans/CRC0327_NT_2/CRC0327_NT_2_kmeans_2comp.csv',
            '/mnt/cold1/snaketree/prj/scRNA/dataset/KMeans_2/kmeans/CRC0542_NT72h_1/CRC0542_NT72h_1_kmeans_2comp.csv')

load <- function(file) {
  df <- read.csv(file, sep=',', header=TRUE, row.names=1)
  
}
alld <- data.frame()
cells <- colnames(sdata)

for (i in seq(1, length(filesp))) {
    data <- read.csv(filesp[i], sep=',', header=TRUE, row.names=1)
    data$cellname <- paste0(rownames(data), '_', i)
    data[data$isPaneth == "filtered", 'isPaneth'] <- 'nPaneth'            
    alld <- rbind(alld, data)
}



In [ ]:
wo <- colnames(sdata)

alld <- alld[match(wo, alld$cellname),]
all(wo == alld$cellname)
sdata$Paneth <- alld$isPaneth
DimPlot(sdata, reduction = "umap", group.by='Paneth')
table(sdata$Paneth, sdata$seurat_clusters)
